# Day 24: Monitoring and Observability

Logs tell you what happened on one request. Metrics tell you how the whole service is doing, over time. Traces follow one request across every stage it passed through. This notebook builds all three for Day 22's RagService, without ever editing RagService itself.

## Metrics registry

A tiny Counter, Gauge, and Histogram library, rendered in the real Prometheus text format.

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())
from metrics_registry import Counter, Histogram, MetricsRegistry

registry = MetricsRegistry()
requests_total = registry.register(Counter('requests_total', 'Total requests.'))
latency_ms = registry.register(Histogram('latency_ms', 'Request latency.'))
requests_total.inc({'status': 'ok'})
requests_total.inc({'status': 'ok'})
latency_ms.observe(12.5)
latency_ms.observe(340.0)
print(registry.render_prometheus())


## Instrumented service

RagService wrapped from the outside: every query() call records request counts, cache hits, and per-stage latency, with RagService itself unchanged.

In [ ]:
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'day-22-production-rag'))
from config import Config
from instrumented_service import InstrumentedRagService

instrumented = InstrumentedRagService(Config())
instrumented.query('What is RAG?')
instrumented.query('What is RAG?')
print('requests_total(ok):', instrumented.requests_total.value({'status': 'ok'}))
print('cache hits:', instrumented.cache_results_total.value({'result': 'hit'}))


## Metrics endpoint

GET /metrics/prometheus added on top of Day 23's api_server.py, without editing that file. The server starts on a free port and shuts down cleanly.

In [ ]:
import threading, urllib.request
from http.server import ThreadingHTTPServer
from metrics_endpoint import MetricsRequestHandler, build_instrumented_app_state

MetricsRequestHandler.app_state = build_instrumented_app_state()
httpd = ThreadingHTTPServer(('127.0.0.1', 0), MetricsRequestHandler)
port = httpd.server_port
threading.Thread(target=httpd.serve_forever, daemon=True).start()

svc = MetricsRequestHandler.app_state['service']
svc.query('What is RAG?')

req = urllib.request.Request(f'http://127.0.0.1:{port}/metrics/prometheus', method='GET')
with urllib.request.urlopen(req, timeout=5) as resp:
    print('status:', resp.status)
    print(resp.read().decode()[:400])

httpd.shutdown()
httpd.server_close()
print('Server shut down cleanly.')


## Percentiles

Why p95 and p99 matter more than an average: a mean blends a handful of slow outliers into a much larger number of fast requests and barely moves.

In [ ]:
from percentile_tracking import nearest_rank_percentile, mean
latencies = [20 + (i % 15) for i in range(97)] + [850, 920, 1100]
print('mean:', round(mean(latencies), 2))
print('p50:', nearest_rank_percentile(latencies, 50))
print('p95:', nearest_rank_percentile(latencies, 95))
print('p99:', nearest_rank_percentile(latencies, 99))


## Traffic simulation and alerts

300 queries across three phases: normal, a spike, then a slow generator. Metrics sampled every 50 requests, and alert_rules.py checked against the slow phase's readings.

In [ ]:
import traffic_simulation
traffic_simulation.main()


## Log correlation

A request id travels with the request from the log line all the way to a full stage-by-stage trace, linking logs, metrics, and traces together.

In [ ]:
import log_correlation
log_correlation.main()


## Dashboard sketch

A plain text dashboard: panels for traffic, error rate, latency, cache hit rate, and status breakdown, each answering one specific question.

In [ ]:
import dashboard_sketch
dashboard_sketch.main()


## Key takeaways

- Logs, metrics, and traces answer different questions: what happened once, how is the whole service doing over time, and where did the time go in one specific request.
- The four signals, latency, traffic, errors, and saturation, cover almost everything worth watching about a running service.
- A scraper pulling metrics on its own schedule is more robust than a service trying to push them, since a dead service simply fails the next scrape instead of needing to remember to push anything.
- Alerts should fire on symptoms users actually feel, like error rate and p95 latency, not on every internal fluctuation, or people learn to ignore them.
- Percentiles reveal a slow tail that an average hides completely, sometimes even while the average barely moves.
- A request id is what lets you move between a metric's general signal and one specific request's full story.